# <span style="color:darkblue"> Lecture 10c: Analyzing Experiments </span>

<font size = "5">



# <span style="color:darkblue"> I. Import Libraries </span>


<font size = "5">

Install `maketables` in your notebook environment with `pip install maketables` if it is not already installed.

In [10]:
# The "pandas" library is used for processing datasets
# The "numpy" is for numeric observations and random numbers
# The "matplotlib.pyplot" library is for creating graphs

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# We will "alias" two sublibraries in "statsmodels"
# "statsmodels.formula.api" contains functions to estimate models
# "statsmodels.api" contains general-use statistical options

import statsmodels.api as sm
import statsmodels.formula.api as smf
import maketables as mt

<font size = "5">

Import data

In [11]:
dataset = pd.read_stata("data_raw/malawiexperiment.dta")

# <span style="color:darkblue"> II. Context </span>


<font size = "5">

Today we will review a paper by Rebecca Dizon-Ross published <br>
in the American Economic Review (2019).

- In this study, researchers partnered with local schools in Malawi <br>
- This study evaluated the impacts of information about children's <br>
 academic performance on parents subsequent investments in their <br>
  children's education.


https://www.povertyactionlab.org/evaluation/effects-student-performance-information-parental-decision-making-malawi?lang=fr

https://www.nber.org/papers/w24610

<font size = "5">

Intervention

- Parents in Malawi with low literacy levels had trouble interpreting <br>
school report cards.Many parents were unaware that their children were <br>
struggling with school.
- The intervention altered the way that the schools engaged with <br>
 parents. It had an impact on reducing the information gaps.


<font size = "5">

Experimental Design

- Students were assigned to randomly to treatment and control <br>
with 50% probability
- The random assignment was done at the household level

<font size = "5">

Findings


<img src="figures/treatmenteffects_dizonross.png" alt="drawing" width="650"/>



<font size = "5">

At baseline (before the intervention)

- The graph on the left shows parental beliefs at baseline
- Parents of low performing students thought that they were doing <br>
better than they were. Ideally it should  be along the 45-degree line
- Similar results for treated and control groups at baseline because <br>
of randomization

At endline (after the intervention)

- Treated parents had more accurate perceptions of their children's <br>
performance
- The treatment effects varied depending on the baseline test scores


# <span style="color:darkblue"> II. Basic Descriptive Analysis </span>


<font size = "5">

Total number of children

In [12]:
len(dataset)

5268

<font size = "5">

Total number of households

- 2 children per households

In [13]:
unique_ids = pd.unique(dataset["hhid"])
len(unique_ids)
#pd unique brings in all distinct households by hhid - household id
#all_ids = dataset["hhid"]  - this would give me all of the values 
#all_ids = dataset["hhid"].tolist() - this would give me a python list
#dataset["hhid"].tolist() - .tolist() converts a pandas Series (a column) into a regular Python list. It keeps all the values, including duplicates.

2634

<font size = "5">

Calculate number of treated and control

In [14]:
table = pd.crosstab(index = dataset['treat'],columns = "count")
table
#pd.crosstab() makes a frequency table: it counts how often each category appears.

col_0,count
treat,
Control,2654
Treatment,2614


# <span style="color:darkblue"> III. Testing Covariate Balance </span>


<font size = "5">

Subset treated and control observations

In [15]:
dataset_treated = dataset.query('treat == "Treatment"')
dataset_control = dataset.query('treat == "Control"')
# doing this creates new dataframes in order to work with the treatment and control groups separately for analysis purposes

<font size = "5">

Socio-economic information can be collected at baseline <br>
(before the experiment) 

In [16]:
variables_scores      = ["ave"]
variables_respondent  = ["lit","primary_resp_fem","age_par1","farmer"]
variables_household   = ["tot_kids","one_par"]
variables_student     = ["std","age","female","attendance_sv"]

<font size = "5" >

Check that characteristics are similar between treated and control <br>
at baseline

In [17]:
# Compute mean and standard deviation for the treated group
display(dataset_treated[variables_respondent].describe().loc[['mean', 'std']])

# Compute mean and standard deviation for the control group
display(dataset_control[variables_respondent].describe().loc[['mean', 'std']])

#.describe() calculates summary statistics for numeric columns, such as count, mean, standard deviation, minimum, quartiles, and maximum.

#.loc lets you select rows or columns by their labels. It’s usually followed by square brackets, for example:dataset["ave"].describe().loc["mean"]

,lit,primary_resp_fem,age_par1,farmer
mean,0.675613,0.758990,40.974080,0.460587
std,0.468235,0.427778,11.290328,0.498541


,lit,primary_resp_fem,age_par1,farmer
mean,0.667426,0.773926,40.645455,0.465544
std,0.471225,0.418366,10.638597,0.498907


<font size = "5">



<font size = "5" >

Conduct a formal test of whether the coefficients are similar

- We should expect the coefficient on the treatment variable <br>
to be non-significant

In [18]:
reg_model = smf.ols("lit ~ treat ", dataset)
#can you talk about this?? literacy rate impacts treat? 
results = reg_model.fit(cov_type= "HC1")

mt.ETable([results], model_heads=["Baseline balance"]).make(type="gt")
#mt.ETable(...) creates a table using the ETable class from whatever library is imported as mt
#[results] passes results as a one-item list of table results.
#model_heads=["Baseline balance"] sets the table’s heading.
#.make(type="gt") formats the table as a Great Tables (gt) table for display in the notebook
#Great Tables is a Python package for presenting data as polished, readable tables—often with styling, labels, and formatting—in notebooks or reports.

GT(_tbl_data=  __index_level_0__ __index_level_1__                      0
0              coef   treat=Treatment     0.008 <br> (0.013)
1              coef         Intercept  0.667*** <br> (0.009)
2             stats      Observations                  5,242
3             stats                R²               0.000076, _body=<great_tables._gt_data.Body object at 0x000001A3415F3B60>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001A3415DFED0>, _spanners=Spanners([SpannerInfo(spanner_id='lit', spanner_level=2, spanner_label=Html(text='lit'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Baseline balance', spanner_level=1, spanner_label=Html(text='Baseline balance'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001A3453101A0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001A3415DED50>, _source_notes=[Html(text='Significance levels: * p &lt; 0.1, ** p &lt; 0.05, *** p &lt; 0.01. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000001A3453102F0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000001A3453106E0>, <great_tables._gt_data.FormatInfo object at 0x000001A3415DFB10>, <great_tables._gt_data.FormatInfo object at 0x000001A3415DEC10>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value

<font size = "5">

Try it yourself!

<font size = "3">

- Obtain summary statistics for the mean and standard deviation for <br>
the other sets of baseline variables


In [25]:
# Write your own code
display(dataset_treated[variables_household].describe().loc[['mean', 'std']])






,tot_kids,one_par
mean,5.101760,0.198929
std,1.708462,0.399271


In [27]:
display(dataset_treated[variables_student].describe().loc[['mean', 'std']])


,std,age,female,attendance_sv
mean,3.723030,11.568383,0.499208,0.913829
std,1.370438,2.663856,0.500098,0.128671


<font size = "5">

Try it yourself!

<font size = "3">

- Write a loop that runs different regressions of baseline covariates <br>
on the treatment variable. This can help you automate the process of <br>
testing for covariate balance

In [28]:
display(dataset_treated[variables_scores].describe().loc[['mean', 'std']])



,ave
mean,46.357307
std,17.530622


# <span style="color:darkblue"> IV. Calculating Average Treatment Effect </span>


- Make sure to use robust standard errors

<font size = "5">

Effect of treatment on endline beliefs

In [21]:
reg_model = smf.ols("u_ave ~ treat ", dataset)
results = reg_model.fit(cov_type= "HC1")

mt.ETable([results], model_heads=["Treatment effect"]).make(type="gt")

GT(_tbl_data=  __index_level_0__ __index_level_1__                       0
0              coef   treat=Treatment  -7.422*** <br> (0.499)
1              coef         Intercept  63.563*** <br> (0.343)
2             stats      Observations                   5,244
3             stats                R²                   0.041, _body=<great_tables._gt_data.Body object at 0x000001A34160F100>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001A3452EC2B0>, _spanners=Spanners([SpannerInfo(spanner_id='u_ave', spanner_level=2, spanner_label=Html(text='u_ave'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Treatment effect', spanner_level=1, spanner_label=Html(text='Treatment effect'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001A3415DEAD0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001A34160DF30>, _source_notes=[Html(text='Significance levels: * p &lt; 0.1, ** p &lt; 0.05, *** p &lt; 0.01. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000001A3415DF250>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000001A34160DE00>, <great_tables._gt_data.FormatInfo object at 0x000001A34160FE10>, <great_tables._gt_data.FormatInfo object at 0x000001A343D8CA70>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', ty

<font size = "5">

Add baseline covariates



Baseline covariates are characteristics measured before treatment begins—such as a student’s age or prior score.

Adding them to a regression can:

Improve precision: people with similar baseline characteristics may have more similar outcomes, helping the model estimate the treatment effect more precisely.
Account for chance differences: even in a randomized experiment, treatment and control groups can differ a little by chance.
The treatment assignment remains the key factor of interest. Covariates should generally be measured before treatment; adjusting for something changed by the treatment can distort the estimate.

In [22]:
reg_model = smf.ols("u_ave ~ treat + ave ", dataset)
results = reg_model.fit(cov_type= "HC1")

mt.ETable([results], model_heads=["Treatment + baseline control"]).make(type="gt")

GT(_tbl_data=  __index_level_0__ __index_level_1__                       0
0              coef   treat=Treatment  -7.056*** <br> (0.432)
1              coef               ave   0.508*** <br> (0.013)
2              coef         Intercept  39.655*** <br> (0.680)
3             stats      Observations                   5,244
4             stats                R²                   0.273, _body=<great_tables._gt_data.Body object at 0x000001A341643CE0>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001A3416438A0>, _spanners=Spanners([SpannerInfo(spanner_id='u_ave', spanner_level=2, spanner_label=Html(text='u_ave'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Treatment + baseline control', spanner_level=1, spanner_label=Html(text='Treatment + baseline control'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001A34160EFD0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001A343D8EF90>, _source_notes=[Html(text='Significance levels: * p &lt; 0.1, ** p &lt; 0.05, *** p &lt; 0.01. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000001A345308690>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000001A3416429C0>, <great_tables._gt_data.FormatInfo object at 0x000001A341643680>, <great_tables._gt_data.FormatInfo object at 0x000001A3450E9850>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='p

<font size = "5">

Test for heterogeneous effects


In [23]:
reg_model = smf.ols("u_ave ~ treat + ave + treat*ave", dataset)
results = reg_model.fit(cov_type= "HC1")

mt.ETable([results], model_heads=["Heterogeneous effect"]).make(type="gt")

GT(_tbl_data=  __index_level_0__      __index_level_1__                        0
0              coef        treat=Treatment  -25.998*** <br> (1.238)
1              coef                    ave    0.305*** <br> (0.018)
2              coef  treat=Treatment × ave    0.405*** <br> (0.024)
3              coef              Intercept   49.188*** <br> (0.884)
4             stats           Observations                    5,244
5             stats                     R²                    0.309, _body=<great_tables._gt_data.Body object at 0x000001A3450E9250>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000001A346B06120>, _spanners=Spanners([SpannerInfo(spanner_id='u_ave', spanner_level=2, spanner_label=Html(text='u_ave'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None), SpannerInfo(spanner_id='Heterogeneous effect', spanner_level=1, spanner_label=Html(text='Heterogeneous effect'), spanner_units=None, spanner_pattern=None, vars=['0'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000001A341643460>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000001A341643BD0>, _source_notes=[Html(text='Significance levels: * p &lt; 0.1, ** p &lt; 0.05, *** p &lt; 0.01. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000001A3452EC770>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000001A3450E9D50>, <great_tables._gt_data.FormatInfo object at 0x000001A346B05F40>, <great_tables._gt_data.FormatInfo object at 0x000001A346B05C70>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', 

<font size = "5">

Try it yourself!

Test for heterogeneous effects using other baseline covariates!

In [24]:
# Write your own code




